# 05 -- Model Comparison, Diagnostics, and Ablation Study

This notebook provides a comprehensive evaluation of the black hole mass prediction task:

1. **Model comparison** -- Ridge regression, Random Forest, XGBoost, LightGBM, MLP trained and evaluated with 5-fold CV
2. **Predicted vs True plots** for all models side-by-side
3. **Feature importance** for tree-based models
4. **Ablation study** -- quantifies the contribution of redshift (Z_FIT) as a feature
5. **Physical diagnostics** -- rest-frame tau vs MBH and sigma vs MBH correlation plots, compared to literature

**Input:**  `data/DRW_results.csv`
**Outputs:** Comparison tables and diagnostic plots saved to `data/`

**Reference comparisons:**
- Helias et al. 2026: slope=0.32+/-0.03, intercept=104 days at 10^8 Msun (N=127)
- Burke et al. 2021: slope=0.38+0.05/-0.04, intercept=107 days, N=67
- MacLeod et al. 2010: sigma-MBH power-law slope ~0.18


In [ ]:
# !pip install xgboost lightgbm scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from scipy import stats

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

DATA_DIR = Path("data")

## 1. Load and Clean Data

In [ ]:
DATA_PATH = DATA_DIR / "DRW_results.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")

FEATURES = ["tau", "sigma", "mu", "PSFMAG_r", "Z_FIT"]
TARGET   = "LOGMBH"

df_clean = df.dropna(subset=FEATURES + [TARGET]).copy()
bad_mask = (df_clean["tau"] <= 0) | (df_clean["sigma"] <= 0)
df_clean = df_clean[~bad_mask].copy()

# Log-transform
df_clean["log_tau"]   = np.log10(df_clean["tau"])
df_clean["log_sigma"] = np.log10(df_clean["sigma"])

FEATURES_TRANSFORMED = ["log_tau", "log_sigma", "mu", "PSFMAG_r", "Z_FIT"]
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean[~df_clean[FEATURES_TRANSFORMED].isna().any(axis=1)].copy()

X = df_clean[FEATURES_TRANSFORMED].values
y = df_clean[TARGET].values

print(f"Final clean sample: {len(df_clean)} quasars")
print(f"y range: {y.min():.2f} -- {y.max():.2f}")


## 2. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")


## 3. Define Models

In [ ]:
models = {
    "Linear Regression (Ridge)": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  Ridge(alpha=1.0))
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=500, max_depth=6, min_samples_leaf=5,
        random_state=42, n_jobs=-1,
    ),

    "XGBoost": XGBRegressor(
        n_estimators=500, learning_rate=0.02, max_depth=3,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="rmse", early_stopping_rounds=30,
        random_state=42,
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=500, learning_rate=0.02, max_depth=3,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1,
    ),

    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  MLPRegressor(
            hidden_layer_sizes=(128, 64, 32), activation="relu",
            max_iter=500, early_stopping=True, random_state=42,
        ))
    ]),
}


## 4. Train and Evaluate All Models

Each model is evaluated on the held-out test set and also with 5-fold cross-validation.
XGBoost requires an eval_set for early stopping, so it uses a slightly different fit call.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results       = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining: {name}")

    if name == "XGBoost":
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    else:
        model.fit(X_train, y_train)

    trained_models[name] = model

    y_pred = model.predict(X_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)

    # For CV, XGBoost needs a version without early_stopping_rounds
    cv_model = (
        XGBRegressor(n_estimators=500, learning_rate=0.02, max_depth=3,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
        if name == "XGBoost" else model
    )

    cv_r2   = cross_val_score(cv_model, X, y, cv=kf, scoring="r2")
    cv_rmse = cross_val_score(cv_model, X, y, cv=kf, scoring="neg_root_mean_squared_error")

    results.append({
        "Model"          : name,
        "Test R2"        : round(r2, 4),
        "Test RMSE"      : round(rmse, 4),
        "CV R2 (mean)"   : round(cv_r2.mean(), 4),
        "CV R2 (std)"    : round(cv_r2.std(), 4),
        "CV RMSE (mean)" : round((-cv_rmse).mean(), 4),
        "CV RMSE (std)"  : round((-cv_rmse).std(), 4),
    })

    print(f"  Test  -- R2: {r2:.4f}  |  RMSE: {rmse:.4f} dex")
    print(f"  5-CV  -- R2: {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}"
          f"  |  RMSE: {(-cv_rmse).mean():.4f} +/- {(-cv_rmse).std():.4f} dex")

results_df = pd.DataFrame(results)
print("\n" + "="*70)
print(results_df.to_string(index=False))


## 5. Results Table (Rendered)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.axis("off")

table = ax.table(
    cellText=results_df.values,
    colLabels=results_df.columns,
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.auto_set_column_width(col=list(range(len(results_df.columns))))
plt.title("Model Comparison -- Black Hole Mass Prediction", fontsize=12, pad=20)
plt.tight_layout()
plt.savefig(DATA_DIR / "model_comparison_table.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Predicted vs True -- All Models

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes  = axes.flatten()
lims  = [y.min() - 0.2, y.max() + 0.2]

for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred = model.predict(X_test)
    r2_i   = r2_score(y_test, y_pred)
    rmse_i = np.sqrt(mean_squared_error(y_test, y_pred))

    ax.scatter(y_test, y_pred, alpha=0.3, s=8, color="steelblue")
    ax.plot(lims, lims, "r--", lw=1.5, label="1:1")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("True log(MBH/Msun)", fontsize=10)
    ax.set_ylabel("Predicted log(MBH/Msun)", fontsize=10)
    ax.set_title(f"{name}\nRMSE={rmse_i:.3f} dex  |  R2={r2_i:.3f}", fontsize=10)
    ax.legend(fontsize=8)

axes[-1].set_visible(False)   # hide unused subplot
plt.suptitle("Predicted vs True Black Hole Mass -- All Models", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / "model_comparison_plots.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Feature Importance -- Tree-Based Models

In [ ]:
tree_models = ["Random Forest", "XGBoost", "LightGBM"]
fig, axes  = plt.subplots(1, 3, figsize=(14, 4))

for ax, name in zip(axes, tree_models):
    model = trained_models[name]
    # Random Forest wraps in a Pipeline; XGBoost/LightGBM expose importances directly
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    else:
        importances = model.named_steps["model"].feature_importances_

    ax.barh(FEATURES_TRANSFORMED, importances, color="steelblue")
    ax.set_xlabel("Importance")
    ax.set_title(name)

plt.suptitle("Feature Importance -- Tree-based Models", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / "feature_importance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Ablation Study -- Impact of Redshift (Z_FIT)

We retrain XGBoost and LightGBM with and without `Z_FIT` to quantify how much
redshift information contributes to mass prediction accuracy.


In [ ]:
FEATURES_NO_Z = ["log_tau", "log_sigma", "mu", "PSFMAG_r"]
X_no_z = df_clean[FEATURES_NO_Z].values

ablation_results = []

for feat_set, X_ab, label in [
    (FEATURES_NO_Z,        X_no_z, "DRW + PSFMAG_r (no redshift)"),
    (FEATURES_TRANSFORMED, X,      "DRW + PSFMAG_r + Z_FIT"),
]:
    X_tr, X_te, y_tr, y_te = train_test_split(X_ab, y, test_size=0.2, random_state=42)

    for name, ModelClass, kwargs in [
        ("XGBoost",  XGBRegressor,  dict(n_estimators=500, learning_rate=0.02, max_depth=3,
                                          subsample=0.8, colsample_bytree=0.8, random_state=42)),
        ("LightGBM", LGBMRegressor, dict(n_estimators=500, learning_rate=0.02, max_depth=3,
                                          subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)),
    ]:
        m = ModelClass(**kwargs)
        m.fit(X_tr, y_tr)
        y_pred_ab = m.predict(X_te)
        ablation_results.append({
            "Model"   : name,
            "Features": label,
            "R2"      : round(r2_score(y_te, y_pred_ab), 4),
            "RMSE"    : round(np.sqrt(mean_squared_error(y_te, y_pred_ab)), 4),
        })

ablation_df = pd.DataFrame(ablation_results)
print(ablation_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric in zip(axes, ["R2", "RMSE"]):
    x = np.arange(2)
    width = 0.35
    for j, model_name in enumerate(["XGBoost", "LightGBM"]):
        subset = ablation_df[ablation_df["Model"] == model_name]
        ax.bar(x + j * width, subset[metric].values, width, label=model_name, alpha=0.8)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(["No Z_FIT", "+Z_FIT"], fontsize=9)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} -- With vs Without Redshift")
    ax.legend()

plt.suptitle("Ablation Study -- Impact of Redshift Feature", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / "ablation_study.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Physical Diagnostics -- Rest-Frame tau vs Black Hole Mass

We apply reliability cuts to the DRW fits and compare the tau-MBH power-law
slope recovered from our sample against published values from the literature.

**Reliability cuts applied:**
- tau_rest > 10 days (physical lower bound)
- tau_rest < ZTF baseline / 10 ~ 182 days (baseline condition)
- n_epochs > 100 (well-sampled light curves only)


In [ ]:
# Compute rest-frame timescale (corrected for cosmological time dilation)
df_clean["tau_restframe"] = df_clean["tau"] / (1 + df_clean["Z_FIT"])

ZTF_BASELINE_DAYS = 1825   # ~5 years

# Apply epoch and reliability cuts
df_hq = df_clean[df_clean["n_epochs"] > 100].copy()

reliable = (
    (df_hq["tau_restframe"] > 10.0) &
    (df_hq["tau_restframe"] < ZTF_BASELINE_DAYS / 10) &
    (df_hq["tau_restframe"] < 1000)
)
df_fit = df_hq[reliable].copy()
print(f"Objects passing reliability cuts: {len(df_fit)}")

log_tau = np.log10(df_fit["tau_restframe"].values)
log_mbh = df_fit["LOGMBH"].values
log_mbh_norm = log_mbh - 8.0   # normalise to 10^8 Msun (matches Helias+ convention)

slope, intercept, r_value, p_value, std_err = stats.linregress(log_mbh_norm, log_tau)

print(f"\nYour fit (tau-MBH):")
print(f"  Slope     : {slope:.3f} +/- {std_err:.3f}")
print(f"  Intercept : {10**intercept:.1f} days at 10^8 Msun")
print(f"  R         : {r_value:.3f}  (R2={r_value**2:.3f})")
print(f"  p-value   : {p_value:.2e}")
print(f"\nHelias et al. 2026: slope=0.32+/-0.03, intercept=104 days, N=127")
print(f"Burke et al. 2021 : slope=0.38+0.05/-0.04, intercept=107 days, N=67")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(log_mbh, log_tau, alpha=0.15, s=8, color="steelblue",
           label=f"This work (N={len(df_fit)})")

mbh_range = np.linspace(log_mbh.min(), log_mbh.max(), 100)

# Our fit
tau_fit = slope * (mbh_range - 8.0) + intercept
ax.plot(mbh_range, tau_fit, "b-", lw=2,
        label=f"This work: slope={slope:.2f}+/-{std_err:.2f}")

# Literature comparisons
tau_helias = 0.32 * (mbh_range - 8.0) + np.log10(104)
ax.plot(mbh_range, tau_helias, "g--", lw=2,
        label="Helias et al. 2026: slope=0.32+/-0.03")

tau_burke = 0.38 * (mbh_range - 8.0) + np.log10(107)
ax.plot(mbh_range, tau_burke, "r--", lw=2,
        label="Burke et al. 2021: slope=0.38")

ax.set_xlabel("log(MBH/Msun)", fontsize=13)
ax.set_ylabel("log(tau_rest / days)", fontsize=13)
ax.set_title("Rest-Frame DRW Timescale vs Black Hole Mass", fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / "tau_mbh_relation.png", dpi=150)
plt.show()


## 10. Physical Diagnostics -- Sigma vs Black Hole Mass

In [ ]:
log_sigma_fit = np.log10(df_fit["sigma"].values)
log_mbh_norm_fit = log_mbh - 8.0

slope_s, intercept_s, r_s, p_s, stderr_s = stats.linregress(log_mbh_norm_fit, log_sigma_fit)

print(f"sigma-MBH fit:")
print(f"  Slope  : {slope_s:.3f} +/- {stderr_s:.3f}")
print(f"  R      : {r_s:.3f}  (R2={r_s**2:.3f})")
print(f"  p-value: {p_s:.2e}")
print(f"\nMacLeod et al. 2010: sigma slope ~0.18 +/- 0.03")

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(log_mbh, log_sigma_fit, alpha=0.2, s=8, color="steelblue",
           label=f"This work (N={len(df_fit)})")

# Our best-fit line
sigma_fit_line = slope_s * (mbh_range - 8.0) + intercept_s
ax.plot(mbh_range, sigma_fit_line, "b-", lw=2,
        label=f"This work: slope={slope_s:.3f}+/-{stderr_s:.3f}")

# MacLeod et al. 2010 reference (anchored at sample median)
median_log_sigma  = np.median(log_sigma_fit)
median_log_mbh_norm = np.median(log_mbh) - 8.0
macleod_offset = median_log_sigma - 0.18 * median_log_mbh_norm
sigma_macleod  = 0.18 * (mbh_range - 8.0) + macleod_offset
ax.plot(mbh_range, sigma_macleod, "r--", lw=2,
        label="MacLeod et al. 2010: slope=0.18 (anchored at median)")

ax.set_xlabel("log(MBH/Msun)", fontsize=13)
ax.set_ylabel("log(sigma / mag)", fontsize=13)
ax.set_title("DRW Variability Amplitude vs Black Hole Mass", fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

stats_text = f"slope = {slope_s:.3f} +/- {stderr_s:.3f}\nR = {r_s:.3f}, p = {p_s:.3f}"
ax.annotate(stats_text, xy=(0.05, 0.85), xycoords="axes fraction",
            fontsize=10, bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))

plt.tight_layout()
plt.savefig(DATA_DIR / "sigma_mbh_relation.png", dpi=150)
plt.show()
